# 4.2 协同过滤：UserCF 与 ItemCF

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

单独掌握邻域推荐：从真实 MovieLens 评分数据、余弦相似度，到 UserCF/ItemCF 训练、Top-K 推理、HR@10 与 Coverage 评测。

## Setup

本 Notebook 的默认真实数据是 **MovieLens latest-small（smoke 确定性切片）；full 用官方完整 MovieLens latest（约 33M 评分）**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [GroupLens MovieLens](https://grouplens.org/datasets/movielens/) · [Collaborative Filtering](https://dl.acm.org/doi/10.1145/192844.192905)

In [ ]:
from pathlib import Path
import os, sys, json
import torch
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_ROOT = Path(os.environ.get("RECSYS_ARTIFACT_ROOT", PROJECT_ROOT)).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("RECSYS_PROFILE", "full")
PROFILE = os.environ["RECSYS_PROFILE"]
CUDA_AVAILABLE = torch.cuda.is_available()
from recsys_lab.data import (load_movielens, movielens_provenance, load_amazon_2023,
                             amazon_provenance, load_kuairand, kuairand_provenance,
                             load_amazon_2018, amazon_2018_provenance,
                             load_movielens_1m, movielens_1m_provenance,
                             load_mind_amazon_books, mind_amazon_provenance,
                             load_census_income, census_income_provenance)
DATASET_KEY = "movielens"
if DATASET_KEY == "movielens":
    real_ratings, real_movies = load_movielens()
    real_interactions = real_ratings
    REAL_DATASET = movielens_provenance(real_ratings)
elif DATASET_KEY == "amazon-books":
    real_ratings = load_amazon_2018("Books") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "mind-amazon-books":
    real_ratings = load_mind_amazon_books() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = mind_amazon_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "amazon-electronics":
    real_ratings = load_amazon_2018("Electronics") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "movielens-1m":
    real_ratings = load_movielens_1m() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = movielens_1m_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "census-income" and PROFILE == "full":
    census_train_x, census_train_y, census_test_x, census_test_y = load_census_income()
    real_interactions, real_movies, real_ratings = census_train_x, None, census_train_x
    REAL_DATASET = census_income_provenance()
elif DATASET_KEY == "amazon-2023":
    real_ratings = load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_provenance(real_ratings)
else:
    real_interactions, real_movies = load_kuairand()
    real_ratings = real_interactions
    REAL_DATASET = kuairand_provenance(real_interactions)
print({"profile": PROFILE, "project_root": str(PROJECT_ROOT), "artifact_root": str(ARTIFACT_ROOT), "real_dataset": REAL_DATASET,
       "cuda_available": CUDA_AVAILABLE,
       "cuda_device": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

## 来源论文与阅读提示

**Resnick et al. (1994), GroupLens** 将“相似用户的评分加权”落成了可运行系统，是 UserCF 的代表性早期来源；**Sarwar et al. (2001), Item-based Collaborative Filtering Recommendation Algorithms** 则系统比较 item 相似度与预测方式。后者的重要工业含义是：item 关系通常比用户兴趣稳定，可以离线计算 Top-N 近邻表，线上只聚合用户历史。

本实验用二值矩阵突出共现传播。真实评分 UserCF 还可做均值中心化；隐式反馈则常加入热门度惩罚、时间衰减和置信度权重。

### 公式怎么读（直觉版）

若向量、点积或矩阵乘法还陌生，请先运行 **3.0 推荐算法数学基础**。这里的 $R_u\cdot R_v$ 只是把两位用户在每个物品上的 0/1 逐项相乘再相加，因此等于共同喜欢数；分母是两行的“勾股长度”，用于避免行为多的人天然得分更高。后面的 `similarity @ train_matrix` 则是一次批量完成所有邻居加权。

## Steps

## 1. 数据集与评测协议

### 1.1 MovieLens 是什么？

[MovieLens](https://grouplens.org/datasets/movielens/) 是推荐系统最常用的公开教学数据之一。MovieLens 100K 包含用户对电影的 1–5 星评分，核心字段为：

- `user_id`：用户标识；
- `item_id`：电影标识；
- `rating`：显式偏好强度；
- `timestamp`：行为发生时间；
- 电影标题、类型等 metadata。

本 Notebook 使用仓库内固定版本的 **MovieLens latest-small 真实评分**。为控制 CPU 时间，只确定性选取活跃用户和高频电影；所有行仍来自 GroupLens 原始文件，不制造评分或时间戳。这里的数值用于教学回归，不作为统一公开 benchmark 成绩。

### 1.2 为什么按时间切分？

真实推荐只能用过去预测未来。我们把每个用户最后一次行为作为测试目标、其余行为作为训练历史。随机切分会让“未来行为”进入训练集，从而高估效果。

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error, roc_auc_score, log_loss

SEED = 2026
torch.manual_seed(SEED)
from recsys_lab.data import load_movielens, leave_last_out, movielens_provenance

ratings, movies = load_movielens(max_users=48, max_items=360, min_user_events=12)
train_ratings, test_ratings = leave_last_out(ratings)
n_users, n_items = ratings.user_id.nunique(), ratings.item_id.nunique()

print({
    "rows": len(ratings), "users": ratings.user_id.nunique(), "items": ratings.item_id.nunique(),
    "train_rows": len(train_ratings), "test_rows": len(test_ratings),
    "sparsity": round(1 - len(train_ratings) / (n_users * n_items), 3),
    "source": movielens_provenance(ratings)["source"], "fabricated_rows": 0,
})
display(ratings[["userId", "movieId", "rating", "timestamp", "title", "genres"]].head(8))

### 1.3 两组任务、四类指标

1. **Top-K 推荐**：对每位用户生成 K 个未见物品，检查测试物品是否命中。使用 HitRate@K、Recall@K、Coverage。
2. **评分预测**：预测 1–5 星，使用 RMSE。RMSE 越低越好，但它不直接等价于 Top-K 体验。
3. **点击率预估**：预测曝光后是否点击，使用 AUC 与 LogLoss。AUC 衡量排序，LogLoss 同时惩罚错误且过度自信的概率。

下面先定义几个透明、可复用的评测函数。

In [ ]:
def topk_items(score_matrix, seen_matrix, k=10):
    scores = score_matrix.copy()
    scores[seen_matrix > 0] = -np.inf
    return np.argsort(-scores, axis=1)[:, :k]

def hit_rate_at_k(topk, targets):
    return float(np.mean([target in row for row, target in zip(topk, targets)]))

def catalog_coverage(topk, catalog_size):
    return float(len(np.unique(topk)) / catalog_size)

test_targets = test_ratings.sort_values("user_id").item_id.to_numpy()

---

## 2. 协同过滤：直接利用邻域

### 2.1 直觉

- **UserCF**：如果 Alice 与 Bob 过去喜欢的电影相似，那么 Bob 喜欢、Alice 没看过的电影可以推荐给 Alice。
- **ItemCF**：如果《A》和《B》经常被同一批用户喜欢，那么看过《A》的用户可能也喜欢《B》。

UserCF 的邻居会随用户兴趣变化，适合用户关系明显的场景；ItemCF 的物品相似度通常更稳定，且“因为你看过 A”更容易解释，因此电商、视频相关推荐中更常见。这一思想可以追溯到 GroupLens（1994）：论文第 7 页手算过 Ken 与 Lee 的相关系数为 -0.8、与 Meg 为 +1，于是同一篇文章对 Ken 预测 4.56 分、对 Nan 只有 3.75 分——推荐权重完全由“过去是否一致”决定。

### 2.2 数学：余弦相似度与邻域加权

**通用先修：** [3.2 隐式反馈、未知项与假负例](/notebooks/3_2_data_ml_basics#implicit-feedback) · [3.3 逐元素、点积与余弦](/notebooks/3_3_linear_algebra#elementwise-dot) · [3.3 矩阵乘法](/notebooks/3_3_linear_algebra#matmul-embedding)<br>
**本论文新增数学：** GroupLens 的 Pearson 用户相关与均值中心化、Sarwar ItemCF 的 item 相似度/加权预测，以及本教程二值余弦实现之间的对应边界。

先把符号说清楚：

- $U$、$I$ 是用户集合与物品集合；$R \in \{0,1\}^{|U|\times|I|}$ 是二值交互矩阵，$R_{ui}=1$ 表示用户 $u$ 喜欢过物品 $i$。
- $R_u$ 是矩阵的第 $u$ 行，即用户 $u$ 的“喜欢清单”，长度等于物品总数 $|I|$。
- $R_u\cdot R_v$ 是点积：两行逐项相乘再相加。元素只有 0/1，所以结果恰好等于两人**共同喜欢**的物品数。
- $\|R_u\|_2=\sqrt{\sum_i R_{ui}^2}$ 是向量的“勾股长度”，这里等于 $\sqrt{\text{喜欢数}}$；除以它是为了避免“行为多的人天然和谁都很像”。

UserCF 的相似度为：

$$
s(u,v)=\frac{R_u\cdot R_v}{\|R_u\|_2\|R_v\|_2}
$$

对用户 $u$ 和候选物品 $i$，UserCF 分数为：

$$
\hat y_{ui}=\sum_{v\in N_k(u)}s(u,v)R_{vi}
$$

其中 $N_k(u)$ 是与 $u$ 最相似的 $k$ 个邻居；公式的意思是：每个邻居投票“我喜欢 $i$”，票数按相似度加权。

**手算一遍（与 2.3 的玩具矩阵相同）。** 取 3 用户 × 4 物品：u0 = [1,1,0,0]，u1 = [1,0,1,0]，u2 = [0,1,1,1]。

- $s(u_0,u_1)=1/(\sqrt2\cdot\sqrt2)=0.5$（共同喜欢只有 i0）；
- $s(u_0,u_2)=1/(\sqrt2\cdot\sqrt3)\approx0.408$（共同喜欢只有 i1）。

给 u0 的候选打分：i2 得 $0.5\times1+0.408\times1=0.908$，i3 得 $0.5\times0+0.408\times1=0.408$，所以 i2 排在 i3 前面。ItemCF 只是把矩阵转置，在物品邻域中做同样的加权：$s(i,j)$ 用 $R$ 的列（物品的“被喜欢清单”）计算，$\hat y_{ui}=\sum_{j\in N_k(i)}s(i,j)R_{uj}$。实际系统还会加入热门度惩罚、时间衰减和相似度截断。

#### 论文公式与本教程实现怎样对应

本教程为了突出“共同喜欢数”，先把 `rating >= 3` 变成 0/1，再算普通余弦；它不会表达“用户 A 总比用户 B 打分严格”。GroupLens 的早期 UserCF 使用 **Pearson 相关**，先减去每位用户自己的平均评分：

$$
s_{\text{Pearson}}(u,v)=
\frac{\sum_{i\in I_{uv}}(r_{ui}-\bar r_u)(r_{vi}-\bar r_v)}
{\sqrt{\sum_{i\in I_{uv}}(r_{ui}-\bar r_u)^2}\sqrt{\sum_{i\in I_{uv}}(r_{vi}-\bar r_v)^2}}.
$$

$I_{uv}$ 是两人都评分的物品；正相关表示“相对各自习惯，两人会一起打高或一起打低”。预测评分时还要把邻居的偏差加回目标用户均值。Sarwar 的 ItemCF 比较了普通余弦、相关性与 **adjusted cosine**；后者把物品列中的每个评分减去该评分用户的均值，再计算 item–item 余弦，从而消除用户评分尺度差异。三者不能混称：**二值余弦适合本教程的隐式 Top-K 演示；Pearson/均值中心化与 adjusted cosine 属于论文评分预测口径。**

### 2.3 小矩阵演示：共现矩阵怎样变成推荐分数

先不用完整数据，手工构造 3 个用户 × 4 个物品。`R @ R.T` 统计用户共同喜欢多少物品，`R.T @ R` 统计物品被多少用户共同喜欢；余弦归一化消除活跃度/热门度的量纲。最后的矩阵乘法正对应邻域加权公式。

In [ ]:
toy_R = np.array([
    [1, 1, 0, 0],  # u0 看过 i0、i1
    [1, 0, 1, 0],  # u1 看过 i0、i2
    [0, 1, 1, 1],  # u2 看过 i1、i2、i3
], dtype=float)

toy_user_cooccurrence = toy_R @ toy_R.T
toy_item_cooccurrence = toy_R.T @ toy_R
display(pd.DataFrame(toy_R, index=["u0","u1","u2"], columns=["i0","i1","i2","i3"]))
display(pd.DataFrame(toy_user_cooccurrence, index=["u0","u1","u2"], columns=["u0","u1","u2"]))
display(pd.DataFrame(toy_item_cooccurrence, index=["i0","i1","i2","i3"], columns=["i0","i1","i2","i3"]))

def toy_cosine(matrix):
    normalized = matrix / np.maximum(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-12)
    similarity = normalized @ normalized.T
    np.fill_diagonal(similarity, 0)
    return similarity

toy_user_similarity = toy_cosine(toy_R)
toy_item_similarity = toy_cosine(toy_R.T)
toy_usercf_scores = toy_user_similarity @ toy_R
toy_itemcf_scores = toy_R @ toy_item_similarity
toy_usercf_scores[toy_R > 0] = -np.inf
toy_itemcf_scores[toy_R > 0] = -np.inf

print("u0 UserCF 未见物品分数:", {f"i{i}": round(v, 3) for i, v in enumerate(toy_usercf_scores[0]) if np.isfinite(v)})
print("u0 ItemCF 未见物品分数:", {f"i{i}": round(v, 3) for i, v in enumerate(toy_itemcf_scores[0]) if np.isfinite(v)})
assert np.allclose(toy_user_cooccurrence, toy_R @ toy_R.T)
assert np.allclose(toy_item_cooccurrence, toy_R.T @ toy_R)

### 2.3 训练：计算 UserCF / ItemCF 相似度

In [ ]:
# 教学中把 rating >= 3 视为正向隐式行为。
train_matrix = np.zeros((n_users, n_items), dtype=np.float32)
for row in train_ratings.itertuples():
    train_matrix[row.user_id, row.item_id] = float(row.rating >= 3)

def cosine_similarity_rows(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-8
    normalized = matrix / norms
    similarity = normalized @ normalized.T
    np.fill_diagonal(similarity, 0.0)
    return similarity

user_similarity = cosine_similarity_rows(train_matrix)
item_similarity = cosine_similarity_rows(train_matrix.T)

usercf_scores = user_similarity @ train_matrix
itemcf_scores = train_matrix @ item_similarity
print({"user_similarity": user_similarity.shape, "item_similarity": item_similarity.shape})

### 2.4 推理：屏蔽已看物品并取 Top-K

In [ ]:
usercf_top10 = topk_items(usercf_scores, train_matrix, k=10)
itemcf_top10 = topk_items(itemcf_scores, train_matrix, k=10)

cf_metrics = {
    "UserCF HR@10": hit_rate_at_k(usercf_top10, test_targets),
    "ItemCF HR@10": hit_rate_at_k(itemcf_top10, test_targets),
    "UserCF Coverage": catalog_coverage(usercf_top10, n_items),
    "ItemCF Coverage": catalog_coverage(itemcf_top10, n_items),
}
display(pd.Series(cf_metrics, name="value").to_frame())

example_user = 0
print("user 0 的历史 item:", np.where(train_matrix[example_user] > 0)[0].tolist())
print("UserCF 推荐:", usercf_top10[example_user].tolist())
print("ItemCF 推荐:", itemcf_top10[example_user].tolist())
print("真实下一 item:", int(test_targets[example_user]))

### 2.5 结果讨论与边界

观察上表时不要只看命中率：Coverage 低可能表示模型只反复推荐热门物品。本小样本中每位用户历史很短，相似度容易受单个共现影响，这正是 CF 在稀疏数据上的典型弱点。

**优点**：无需训练神经网络；解释直接；增量更新容易；ItemCF 可作为强兜底召回。  
**缺点**：新用户、新物品无邻居；相似度矩阵可能很大；共现继承曝光偏差与热门偏差；无法自然使用文本、图片等内容。

**推理复杂度**：离线相似度可截断为每个实体 Top-N 邻居；线上只聚合用户历史物品的邻居倒排表，避免扫描全库。

## Checks

检查相似度矩阵对角线、已见物品屏蔽、Top-K 范围以及 Coverage。

In [ ]:
assert np.allclose(np.diag(user_similarity), 0)
assert np.allclose(np.diag(item_similarity), 0)
assert 0 <= cf_metrics['UserCF HR@10'] <= 1
assert 0 <= cf_metrics['ItemCF Coverage'] <= 1
print('PASS：UserCF / ItemCF 训练、推理与评测均有效。')

In [ ]:
result_dir = ARTIFACT_ROOT / "results" / "chapter_4"
result_dir.mkdir(parents=True, exist_ok=True)
payload = {"records": [
    {"algorithm":"UserCF", "task":"Top-K", "metric":"HR@10 ↑", "value":cf_metrics["UserCF HR@10"], "secondary_metric":"Coverage", "secondary_value":cf_metrics["UserCF Coverage"], "online_inference":"聚合相似用户的历史", "source_notebook":"4_2_collaborative_filtering"},
    {"algorithm":"ItemCF", "task":"Top-K", "metric":"HR@10 ↑", "value":cf_metrics["ItemCF HR@10"], "secondary_metric":"Coverage", "secondary_value":cf_metrics["ItemCF Coverage"], "online_inference":"聚合历史物品的近邻", "source_notebook":"4_2_collaborative_filtering"},
]}
(result_dir / "collaborative_filtering.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写出章节指标：collaborative_filtering.json")

## Next Steps

加入热门度惩罚、时间衰减、邻居 Top-N 截断，并在 MovieLens 100K 上比较 UserCF 与 ItemCF 的 HR、Coverage 和长尾表现。